# ALGORITMO

In [2]:
import os
import numpy as np
import re
from scipy.sparse import csr_matrix
import kagglehub
import pandas as pd
import joblib
import pyarrow
import fastparquet
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

Scarichiamo il dataset.

In [18]:
def download_dataset(reviews_path, movies_path):
    if not os.path.isfile(reviews_path) or not os.path.isfile(movies_path):
        print("Missing dataset")

        # download the dataset from kaggle
        dataset_path = kagglehub.dataset_download(
            "stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset",
            output_dir="./data/raw"
        )
        print("Path to dataset files:", dataset_path)

    else:
        print("Dataset already downloaded")

Filtriamo per le colonne che interesseranno l'algoritmo.

In [4]:
def create_dataframe(reviews_path, movies_path):
    movies_df = pd.read_csv(movies_path)
    critic_df = pd.read_csv(reviews_path)
    return critic_df, movies_df

In [6]:
def init_dataframe(movies_df, critic_df):

    temp_dataset = pd.merge(
        critic_df,
        movies_df,
        on="rotten_tomatoes_link",
        how="inner"
    )


    column = [
        "critic_name",           # Username
        "rotten_tomatoes_link",  # Film ID
        "review_type",           # Binary score
        "review_score",           # score of the review
    ]
    dataframe = temp_dataset[column].copy()


    # removing null or NaN
    dataframe = dataframe.dropna(subset= ["critic_name", "review_type", "review_score"])


    dataframe['clean_score'] = dataframe['review_score'].apply(parse_binary_score)
    clean_dataframe = dataframe.dropna(subset=['clean_score'])

    return clean_dataframe

Con questa funzione siamo in grado di ottenere i dati che ci servono ma sono ancora in un formato poco utile a fini matematici. Per risolvere il problema prendiamo in considerazione `review_type` e `review_score`. `review_type` è un dato binario, ma possiamo migliorarlo in un intervallo [0,1] usando `review_score` per decidere che peso ha la recensione rispetto alle altre.

In [7]:
def parse_binary_score(score_str):

    # removing null or NaN
    if pd.isna(score_str):
        return np.nan

    # cleaning the string
    score_str = str(score_str).strip().upper()

    letter_mapping = {
        'A+': 1.0,  'A': 0.95, 'A-': 0.9,
        'B+': 0.85, 'B': 0.8,  'B-': 0.75,
        'C+': 0.65, 'C': 0.6,  'C-': 0.55,
        'D+': 0.45, 'D': 0.4,  'D-': 0.35,
        'F': 0.1
    }

    if score_str in letter_mapping:
        return letter_mapping[score_str]

    # reading value like 3/5, 7.5/10, 0.5/4 ecc.
    match_frac = re.match(r"^(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)$", score_str)

    if match_frac:
        numerator = float(match_frac.group(1))
        denominator = float(match_frac.group(2))

        # avoid division by zero and input errors (e.g. 11/10)
        if denominator > 0 and numerator <= denominator:
            return numerator / denominator

    return np.nan

Arrivati a questo punto possiamo eseguire il codice per ottenere un database pulito con informazioni utili

In [19]:
reviews_path = "./data/raw/rotten_tomatoes_critic_reviews.csv"
movies_path = "./data/raw/rotten_tomatoes_movies.csv"

# download the csv from kaggle
download_dataset(reviews_path, movies_path)

# I convert csv into dataframes
critic_df, movies_df = create_dataframe(reviews_path, movies_path)


#Converto i csv in un database pulito
dataframe_rw = init_dataframe(movies_df,critic_df)

Missing dataset


100%|██████████| 77.2M/77.2M [00:04<00:00, 20.0MB/s]

Extracting files...


Path to dataset files: ./data/raw


Verifichiamo i dati ottenuti:

In [14]:
dataframe_rw.head(10)

,critic_name,rotten_tomatoes_link,review_type,review_score,clean_score
3,Ben McEachen,m/0814255,Fresh,3.5/5,0.70
6,Nick Schager,m/0814255,Rotten,1/4,0.25
7,Bill Goodykoontz,m/0814255,Fresh,3.5/5,0.70
8,Jordan Hoffman,m/0814255,Fresh,B,0.80
9,Jim Schembri,m/0814255,Fresh,3/5,0.60
10,Mark Adams,m/0814255,Fresh,4/5,0.80
11,Roger Moore,m/0814255,Rotten,2/4,0.50
12,David Jenkins,m/0814255,Rotten,2/5,0.40
13,Joshua Tyler,m/0814255,Fresh,3/5,0.60
14,Peter Paras,m/0814255,Rotten,C,0.60


Una volta creato il database delle recensioni dobbiamo procedere alla creazione di un database delle caratterstiche dei film

In [15]:
def calculate_matrix_film(movies_df, threshold_film_products=3):

    df_features = movies_df[['rotten_tomatoes_link', 'genres', 'directors']].copy()
    df_features = df_features.fillna('')


    # let's break the directors' strings into lists and "explode" the list to have one director for each line
    directors = df_features['directors'].str.split(',').explode().str.strip()

    # let's count the frequencies: how many times does each director appear?
    count_directors = directors.value_counts()

    # let's create a mathematical "Whitelist": only those who have made >= (threshold_film_products) films
    valid_directors = set(count_directors[count_directors >= threshold_film_products].index)

    # the filter function: if the director is in the whitelist we keep him, otherwise we discard him
    def filter_directors(director_string):
        if not director_string:
            return []
        clean_list = [r.strip() for r in director_string.split(',') if r.strip() in valid_directors]
        return clean_list

    # Let's apply the filter to our dataset
    df_features['directors_list_filtered'] = df_features['directors'].apply(filter_directors)
    df_features['genres_list'] = df_features['genres'].apply(lambda x: [g.strip() for g in x.split(',') if g.strip()])

    # Vectorization setup
    mlb_genres = MultiLabelBinarizer()
    mlb_directors = MultiLabelBinarizer()

    # actual vectorization
    genres_matrix = mlb_genres.fit_transform(df_features['genres_list'])
    directors_matrix = mlb_directors.fit_transform(df_features['directors_list_filtered'])

    df_generi_bin = pd.DataFrame(genres_matrix, columns=mlb_genres.classes_, index=df_features['rotten_tomatoes_link'])
    df_registi_bin = pd.DataFrame(directors_matrix, columns=mlb_directors.classes_, index=df_features['rotten_tomatoes_link'])

    # let's merge the clean matrices
    movies_df_optimized = pd.concat([df_generi_bin, df_registi_bin], axis=1)

    return movies_df_optimized

Adesso possiamo chiamare la funzione è verificare i risultati ottenuti:

In [16]:
content_matrix = calculate_matrix_film(movies_df)
content_matrix.head(5)

,Action & Adventure,Animation,Anime & Manga,Art House & International,Classics,Comedy,Cult Movies,Documentary,Drama,Faith & Spirituality,...,Yves Robert,Zach Braff,Zack Snyder,Zak Hilditch,Zhang Yimou,Zhangke Jia,Ziad Doueiri,Zoltan Korda,Zoya Akhtar,Álex de la Iglesia
rotten_tomatoes_link,,,,,,,,,,,,,,,,,,,,,
m/0814255,1,0,0,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
m/0878835,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
m/10,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
m/1000013-12_angry_men,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
m/1000079-20000_leagues_under_the_sea,1,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


A questo punto dobbiamo inziare a calcolare la Coseno di Similarità per iniziare a predirre qualcosa.

In [10]:
def calcola_similarita_film(matrice_features):
    print("Accensione motore geometrico...")
    print("Calcolo del prodotto scalare tra 17.712 vettori a 1900 dimensioni in corso...")

    # Questa singola riga esegue milioni di moltiplicazioni matriciali in pochi secondi
    matrice_sim = cosine_similarity(matrice_features)

    # Usiamo il link dei film sia per le righe che per le colonne
    df_similarita = pd.DataFrame(
        matrice_sim,
        index=matrice_features.index,
        columns=matrice_features.index
    )

    print(f"Matrice di similarità creata! Dimensioni: {df_similarita.shape}")
    return df_similarita

In [11]:
matrice_similarita_film = calcola_similarita_film(matrice_contenuti)

Accensione motore geometrico...
Calcolo del prodotto scalare tra 17.712 vettori a 1900 dimensioni in corso...
Matrice di similarità creata! Dimensioni: (17712, 17712)


In [12]:
def raccomanda_film_simili(titolo_film, movies_df, matrice_sim, top_n=5):
    # 1. Troviamo il link ID del film partendo dal titolo (per renderlo human-readable)
    try:
        # Prendiamo il primo risultato che matcha col titolo esatto
        film_id = movies_df[movies_df['movie_title'] == titolo_film]['rotten_tomatoes_link'].iloc[0]
    except IndexError:
        return "Errore: Film non trovato nel database. Controlla le maiuscole o il titolo esatto."

    # 2. Estraiamo la riga delle similarità per quel film specifico
    vettore_similarita = matrice_sim.loc[film_id]

    # 3. Ordiniamo i valori dal più grande al più piccolo e scartiamo il primo (che sarà il film stesso con similarità 1.0)
    film_simili_ids = vettore_similarita.sort_values(ascending=False).iloc[1:top_n+1]

    # 4. Traduciamo gli ID incomprensibili di nuovo in titoli leggibili e stampiamo la percentuale
    print(f"🎬 Perché hai guardato '{titolo_film}', ti consigliamo:")
    print("-" * 50)
    for index, score in film_simili_ids.items():
        titolo_suggerito = movies_df[movies_df['rotten_tomatoes_link'] == index]['movie_title'].iloc[0]
        # Trasformiamo il punteggio decimale in una percentuale facile da leggere
        print(f"[{score*100:.1f}% Match] -> {titolo_suggerito}")


Adesso possiamo cercare film che hanno caratterstiche simili anche se non centrano uno con l'altro

In [13]:
raccomanda_film_simili("Alien", movies_df, matrice_similarita_film)

🎬 Perché hai guardato 'Alien', ti consigliamo:
--------------------------------------------------
[86.6% Match] -> Ex Machina
[86.6% Match] -> Rememory
[86.6% Match] -> Legend
[86.6% Match] -> The Lazarus Project
[86.6% Match] -> Errors Of The Human Body


In [14]:

def costruisci_cervello_collaborativo_sparse(df_voti):
    print("Inizio costruzione architettura sparsa...")

    # 1. Trasformiamo i nomi (testo) in ID numerici (indici)
    # Fondamentale per la mappatura della matrice
    df_voti['critic_id'] = pd.Categorical(df_voti['critic_name']).codes
    df_voti['movie_id'] = pd.Categorical(df_voti['rotten_tomatoes_link']).codes

    # 2. Creiamo la Matrice Sparsa (CSR)
    # Non occupa quasi nulla in RAM perché ignora i vuoti
    matrice_sparsa = csr_matrix((
        df_voti['clean_score'],
        (df_voti['movie_id'], df_voti['critic_id'])
    ))

    print(f"Matrice sparsa creata: {matrice_sparsa.shape}")

    # 3. SVD direttamente sulla matrice sparsa
    # Scikit-learn è intelligente: se gli dai una CSR, usa algoritmi molto più veloci
    print("Compressione SVD in corso...")
    svd = TruncatedSVD(n_components=50, random_state=42)
    matrice_compressa = svd.fit_transform(matrice_sparsa)

    # 4. Similarità sulla matrice compressa (ora è piccola e densa, va bene così)
    matrice_sim_collab = cosine_similarity(matrice_compressa)

    # 5. Mappiamo i risultati ai link dei film
    movie_links = pd.Categorical(df_voti['rotten_tomatoes_link']).categories
    df_sim_final = pd.DataFrame(matrice_sim_collab, index=movie_links, columns=movie_links)

    print("Sistema Collaborativo pronto (e la RAM è salva).")
    return df_sim_final



In [15]:
matrice_similarita_umana = costruisci_cervello_collaborativo_sparse(clean_dataset)


# Fusiome dei due mondi: 50% quello che dice la critica, 50% com'è fatto il film
# Assicurati che entrambe le matrici abbiano gli stessi indici (i film comuni)
film_comuni = matrice_similarita_film.index.intersection(matrice_similarita_umana.index)

matrice_ibrida = (matrice_similarita_film.loc[film_comuni, film_comuni] +
                  matrice_similarita_umana.loc[film_comuni, film_comuni]) / 2



Inizio costruzione architettura sparsa...
Matrice sparsa creata: (17666, 6884)
Compressione SVD in corso...
Sistema Collaborativo pronto (e la RAM è salva).


In [16]:
raccomanda_film_simili("Jurassic Park", movies_df, matrice_ibrida)

🎬 Perché hai guardato 'Jurassic Park', ti consigliamo:
--------------------------------------------------
[79.2% Match] -> Close Encounters of the Third Kind
[72.6% Match] -> Lockout
[71.8% Match] -> Minority Report
[71.7% Match] -> Star Trek VI - The Undiscovered Country
[71.5% Match] -> Jaws


In [17]:

def salva_stato_sistema(movies_df, matrice_ibrida):
    print("Inizio esportazione del cervello del sistema...")

    # 1. Salva il DataFrame dei film in formato Parquet
    # È fondamentale per avere i titoli e i link pronti all'uso
    movies_df.to_parquet('film_database.parquet', engine='fastparquet')


    # 2. Salva la matrice ibrida finale
    # Usiamo joblib con compressione per risparmiare spazio
    joblib.dump(matrice_ibrida, 'matrice_ibrida.joblib', compress=3)

    print("✅ Stato salvato con successo: 'film_database.parquet' e 'matrice_ibrida.joblib' sono pronti.")



In [19]:
# Esegui il salvataggio
salva_stato_sistema(movies_df, matrice_ibrida)

Inizio esportazione del cervello del sistema...


ValueError: Error converting column "rotten_tomatoes_link" to bytes using encoding UTF8. Original error: Unable to avoid copy while creating an array as requested.

In [ ]:
def ripristina_sistema():
    print("📂 Caricamento dati dal disco...")

    # Carichiamo il database film usando fastparquet
    df_film_load = pd.read_parquet('film_database.parquet', engine='fastparquet')

    # Carichiamo la matrice pre-calcolata
    matrice_load = joblib.load('matrice_ibrida.joblib')

    print(f"Sistema ripristinato! Database: {len(df_film_load)} film.")
    return df_film_load, matrice_load

# Quando riparti da zero, usa questa riga:
# movies_df, matrice_ibrida = ripristina_sistema()
